In [0]:
# ============================================================
# WHAT WENT WRONG, AND WHAT THIS ANSWERS
#
# The Online Picking reports match on PAYLOAD_AREACODE = 'Online Picking' and
# on attributes equal to 'Zone: D', 'Zone: F' and so on. Both are EXACT string
# matches, so either one being spelled differently in the source - a different
# area name, 'Zone:D' with no space, a lower-case zone letter, a zone carried
# in some other field - returns nothing at all rather than returning less.
#
# So this prints the raw distinct values for one operator in one 15-minute
# window and lets the source say what it actually holds. Nothing here writes
# anywhere; it is safe to run at any time.
# ============================================================

dbutils.widgets.text("on_date", "12/08/2026", "Date (dd/MM/yyyy)")
dbutils.widgets.text("from_time", "03:15", "From Time (HH:mm, UK local)")
dbutils.widgets.text("to_time", "03:30", "To Time (HH:mm, UK local)")
dbutils.widgets.text("bonus", "2Qp", "Bonus Number")
dbutils.widgets.text("warehouse", "X", "Warehouse Code")

from datetime import datetime
from zoneinfo import ZoneInfo

on_date = dbutils.widgets.get("on_date").strip()
from_time = dbutils.widgets.get("from_time").strip()
to_time = dbutils.widgets.get("to_time").strip()
bonus = dbutils.widgets.get("bonus").strip()
warehouse = dbutils.widgets.get("warehouse").strip()

uk = ZoneInfo("Europe/London")
start_local = datetime.strptime(f"{on_date} {from_time}", "%d/%m/%Y %H:%M").replace(tzinfo=uk)
end_local = datetime.strptime(f"{on_date} {to_time}", "%d/%m/%Y %H:%M").replace(tzinfo=uk)
if end_local <= start_local:
    raise ValueError("To time must be after From time")

# The table stores UTC. In August the UK is on BST, so 03:15 local is 02:15 UTC
# - comparing local times against it directly would read the wrong window.
start_utc = start_local.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S")
end_utc = end_local.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S")

# Matched the way the pipeline canonicalises it, so the case typed in the
# widget cannot be the reason nothing comes back.
bonus_sql = bonus.upper().replace("'", "''")

print(f"Window (UK local): {start_local:%d/%m/%Y %H:%M} -> {end_local:%d/%m/%Y %H:%M}")
print(f"Window (UTC)     : {start_utc} -> {end_utc}")
print(f"Bonus            : {bonus!r}  (matched as {bonus.upper()!r})")
print(f"Warehouse        : {warehouse!r}")

In [0]:
df = (spark.read
      .format("delta")
      .load("abfss://landing@whsanalyticsdlsprodeuw.dfs.core.windows.net/streaming/landing_bonushub_event_parsed/delta/"))

df.createOrReplaceTempView("landing_bonus_hub_event_parsed")
print("View created.")

In [0]:
# ============================================================
# THE ANSWER: every distinct (work area, attribute) this operator produced in
# the window.
#
# Attributes are exploded, so each attribute value gets its own row and its
# exact text is visible - spaces, case and all. An event carrying no attributes
# still appears, as '(no attributes)', because "the area is there but the
# attributes are empty" and "the area is not there at all" are very different
# answers and collapsing them would hide the difference.
# ============================================================

unique_area_attr = spark.sql(f"""
  SELECT
    PAYLOAD_AREACODE                     AS work_area,
    COALESCE(attribute, '(no attributes)') AS attribute,
    COUNT(*)                             AS events
  FROM landing_bonus_hub_event_parsed
  LATERAL VIEW OUTER explode(PAYLOAD_ATTRIBUTES) t AS attribute
  WHERE TRIM(PAYLOAD_WAREHOUSECODE) = '{warehouse}'
    AND upper(trim(PAYLOAD_BONUSCODE)) = '{bonus_sql}'
    AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) >= timestamp('{start_utc}')
    AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) <  timestamp('{end_utc}')
  GROUP BY PAYLOAD_AREACODE, attribute
  ORDER BY work_area, attribute
""")

print(f"{unique_area_attr.count()} distinct (work area, attribute) pair(s)")
display(unique_area_attr)

In [0]:
# ============================================================
# An empty result above means the window/bonus/warehouse combination matched
# no events at all, which is a different problem from a spelling mismatch. This
# widens the net one step at a time so the failing condition names itself.
# ============================================================

base_where = f"""to_timestamp(PAYLOAD_EVENTTIMESTAMP) >= timestamp('{start_utc}')
  AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) <  timestamp('{end_utc}')"""

checks = {
    "window only (any warehouse, any bonus)": base_where,
    "window + warehouse": base_where + f"\n  AND TRIM(PAYLOAD_WAREHOUSECODE) = '{warehouse}'",
    "window + warehouse + bonus": base_where + f"\n  AND TRIM(PAYLOAD_WAREHOUSECODE) = '{warehouse}'"
                                               f"\n  AND upper(trim(PAYLOAD_BONUSCODE)) = '{bonus_sql}'",
}

for label, where in checks.items():
    n = spark.sql(f"SELECT COUNT(*) AS n FROM landing_bonus_hub_event_parsed WHERE {where}").collect()[0]["n"]
    print(f"{n:>10,}  {label}")

In [0]:
# Which area codes exist at all in this window, across everyone. If 'Online
# Picking' is spelled differently at source, it shows up here - and if it is
# absent entirely, the area is not being written under any name.

display(spark.sql(f"""
  SELECT
    PAYLOAD_AREACODE  AS work_area,
    COUNT(*)          AS events,
    COUNT(DISTINCT upper(trim(PAYLOAD_BONUSCODE))) AS operators
  FROM landing_bonus_hub_event_parsed
  WHERE TRIM(PAYLOAD_WAREHOUSECODE) = '{warehouse}'
    AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) >= timestamp('{start_utc}')
    AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) <  timestamp('{end_utc}')
  GROUP BY PAYLOAD_AREACODE
  ORDER BY events DESC
"""))

In [0]:
# The zone predicate is an exact match on 'Zone: D' and friends. This shows
# every attribute that mentions a zone in any casing or spacing, so the real
# format is visible rather than assumed. If these come back as, say, 'ZONE:D'
# or 'Zone D', the report configs need to match that instead.

display(spark.sql(f"""
  SELECT
    attribute,
    COUNT(*) AS events,
    COUNT(DISTINCT PAYLOAD_AREACODE) AS areas
  FROM landing_bonus_hub_event_parsed
  LATERAL VIEW explode(PAYLOAD_ATTRIBUTES) t AS attribute
  WHERE TRIM(PAYLOAD_WAREHOUSECODE) = '{warehouse}'
    AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) >= timestamp('{start_utc}')
    AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) <  timestamp('{end_utc}')
    AND upper(attribute) LIKE '%ZONE%'
  GROUP BY attribute
  ORDER BY events DESC
"""))

In [0]:
# The unaggregated events behind the first cell - event types, quantities and
# the full attribute array as stored. Useful for confirming PickItemEvent is
# the volume event, and for seeing whether the zone travels in the attribute
# array at all or somewhere else entirely.

display(spark.sql(f"""
  SELECT
    from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP), 'Europe/London') AS event_time_uk,
    PAYLOAD_AREACODE      AS work_area,
    PAYLOAD_EVENTTYPE     AS event_type,
    PAYLOAD_ATTRIBUTES    AS attributes_raw,
    PAYLOAD_QUANTITY      AS qty,
    PAYLOAD_STANDARDHOURS AS std_hours,
    PAYLOAD_SMV           AS smv
  FROM landing_bonus_hub_event_parsed
  WHERE TRIM(PAYLOAD_WAREHOUSECODE) = '{warehouse}'
    AND upper(trim(PAYLOAD_BONUSCODE)) = '{bonus_sql}'
    AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) >= timestamp('{start_utc}')
    AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) <  timestamp('{end_utc}')
  ORDER BY event_time_uk
"""))